# Sisko Models

Fit Sisko curves to slurry viscosity measurements and build a row-level physics dataset for downstream modelling.

## 1. Import Libraries

In [1]:
import warnings
import numpy as np
import pandas as pd

from scipy.optimize import least_squares
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

## 2. Load Data

In [2]:
df = pd.read_csv('../data/processed/combined_slurry_data_expanded.csv')

print(f"Rows: {len(df)}  |  Unique formulations (Composite_Mix_ID): {df['Composite_Mix_ID'].nunique()}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Rows: 178  |  Unique formulations (Composite_Mix_ID): 68
Columns: ['Dispersent_Type', 'Solid_Content_pct', 'Solid_Additive_pct', 'Viscosity_at_shear_rate_1_1/s', 'Viscosity_at_shear_rate_10_1/s', 'Viscosity_at_shear_rate_100_1/s', 'Source_Batch', 'Composite_Mix_ID', 'NMC_pct', 'C65_pct', 'KS6L_pct', 'PVDF_pct']


,Dispersent_Type,Solid_Content_pct,Solid_Additive_pct,Viscosity_at_shear_rate_1_1/s,Viscosity_at_shear_rate_10_1/s,Viscosity_at_shear_rate_100_1/s,Source_Batch,Composite_Mix_ID,NMC_pct,C65_pct,KS6L_pct,PVDF_pct
0,Hypermer,73.0,0.50,10.56640,3.78921,1.99755,Batch_1,F1_Hypermer_73.0_0.5,96.0,2.0,0.0,2.0
1,Hypermer,73.0,0.25,71.65190,14.08460,4.82515,Batch_1,F1_Hypermer_73.0_0.25,96.0,2.0,0.0,2.0
2,Hypermer,73.0,0.25,9.64639,3.26827,1.61720,Batch_1,F1_Hypermer_73.0_0.25,96.0,2.0,0.0,2.0
3,Hypermer,77.0,0.25,61.11070,18.77450,7.51220,Batch_1,F1_Hypermer_77.0_0.25,96.0,2.0,0.0,2.0
4,Hypermer,73.0,0.25,8.37111,4.83186,2.30422,Batch_1,F1_Hypermer_73.0_0.25,96.0,2.0,0.0,2.0


## 3. Define Sisko Model

In [3]:
def sisko_viscosity(shear_rate, eta_inf_sisko, k_sisko, n_sisko):
    """
    Sisko viscosity model: eta = eta_inf + K * gamma_dot^(n - 1).

    Parameters
    ----------
    shear_rate     : array-like  -- applied shear rate (1/s)
    eta_inf_sisko  : float       -- infinite-shear viscosity
    k_sisko        : float       -- consistency-like coefficient
    n_sisko        : float       -- flow index (<1 shear-thinning)
    """
    return eta_inf_sisko + k_sisko * np.power(shear_rate, n_sisko - 1.0)

## 4. Set Up Columns and Features

In [4]:
shear_rates = np.array([1.0, 10.0, 100.0], dtype=float)

target_cols = [
    'Viscosity_at_shear_rate_1_1/s',
    'Viscosity_at_shear_rate_10_1/s',
    'Viscosity_at_shear_rate_100_1/s',
]

eps = 1e-8
df['Solid_to_Liquid_Ratio'] = df['Solid_Content_pct'] / (100.0 - df['Solid_Content_pct'] + eps)
df['Binder_to_Active_Ratio'] = df['PVDF_pct'] / (df['NMC_pct'] + df['C65_pct'] + eps)

input_cols = [
    'Dispersent_Type', 'Solid_Content_pct', 'Solid_Additive_pct',
    'NMC_pct', 'C65_pct', 'KS6L_pct', 'PVDF_pct', 'Source_Batch',
    'Solid_to_Liquid_Ratio', 'Binder_to_Active_Ratio',
]

print('Target columns:', target_cols)
print('Input columns :', input_cols)
display(df[input_cols + ['Composite_Mix_ID']].head())

Target columns: ['Viscosity_at_shear_rate_1_1/s', 'Viscosity_at_shear_rate_10_1/s', 'Viscosity_at_shear_rate_100_1/s']
Input columns : ['Dispersent_Type', 'Solid_Content_pct', 'Solid_Additive_pct', 'NMC_pct', 'C65_pct', 'KS6L_pct', 'PVDF_pct', 'Source_Batch', 'Solid_to_Liquid_Ratio', 'Binder_to_Active_Ratio']


,Dispersent_Type,Solid_Content_pct,Solid_Additive_pct,NMC_pct,C65_pct,KS6L_pct,PVDF_pct,Source_Batch,Solid_to_Liquid_Ratio,Binder_to_Active_Ratio,Composite_Mix_ID
0,Hypermer,73.0,0.50,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,F1_Hypermer_73.0_0.5
1,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,F1_Hypermer_73.0_0.25
2,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,F1_Hypermer_73.0_0.25
3,Hypermer,77.0,0.25,96.0,2.0,0.0,2.0,Batch_1,3.347826,0.020408,F1_Hypermer_77.0_0.25
4,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,F1_Hypermer_73.0_0.25


## 5. Fit Sisko Parameters Per Row

In [11]:
USE_LOCKED_ETA_INF_ABLATION = False
LOCKED_ETA_INF_SISKO = 0.01
USE_LOG_RESIDUALS_FOR_SISKO_FIT = True
RESIDUAL_EPS = 1e-8

sisko_dataset = []
failed_fits = []

if USE_LOCKED_ETA_INF_ABLATION:
    def fit_row_sisko_model(x_values, y_values, initial_guess_free, bounds_free, reg_weight=1e-2):
        def residuals(theta_free):
            k_sisko = theta_free[0]
            n_sisko = theta_free[1]
            y_hat = sisko_viscosity(x_values, LOCKED_ETA_INF_SISKO, k_sisko, n_sisko)

            if USE_LOG_RESIDUALS_FOR_SISKO_FIT:
                data_resid = np.log(y_values + RESIDUAL_EPS) - np.log(y_hat + RESIDUAL_EPS)
            else:
                data_resid = y_values - y_hat

            reg_resid = np.sqrt(reg_weight) * (theta_free - initial_guess_free)
            return np.concatenate([data_resid, reg_resid])

        return least_squares(residuals, x0=initial_guess_free, bounds=bounds_free, max_nfev=20000)
else:
    def fit_row_sisko_model(x_values, y_values, initial_guess, bounds, reg_weight=1e-2):
        def residuals(theta):
            eta_inf_sisko = theta[0]
            k_sisko = theta[1]
            n_sisko = theta[2]
            y_hat = sisko_viscosity(x_values, eta_inf_sisko, k_sisko, n_sisko)

            if USE_LOG_RESIDUALS_FOR_SISKO_FIT:
                data_resid = np.log(y_values + RESIDUAL_EPS) - np.log(y_hat + RESIDUAL_EPS)
            else:
                data_resid = y_values - y_hat

            reg_resid = np.sqrt(reg_weight) * (theta - initial_guess)
            return np.concatenate([data_resid, reg_resid])

        return least_squares(residuals, x0=initial_guess, bounds=bounds, max_nfev=20000)

mode_label = 'locked eta_inf=0.01' if USE_LOCKED_ETA_INF_ABLATION else 'full 3-parameter fit'
print(f'Sisko fit mode: {mode_label}')
print(f'Log residuals: {USE_LOG_RESIDUALS_FOR_SISKO_FIT}')
print(f'Fitting Sisko parameters for {len(df)} rows (entire dataset)...')

for row_idx, raw_row in df.iterrows():
    x_data = shear_rates
    y_data = raw_row[target_cols].values.astype(float)

    try:
        if USE_LOCKED_ETA_INF_ABLATION:
            initial_guess_free = np.array([np.max(y_data), 0.5], dtype=float)
            bounds_free = (
                np.array([1e-4, 0.05], dtype=float),
                np.array([1e5, 1.5], dtype=float),
            )

            final_fit = fit_row_sisko_model(x_data, y_data, initial_guess_free, bounds_free)
            eta_inf_sisko = LOCKED_ETA_INF_SISKO
            k_sisko = final_fit.x[0]
            n_sisko = final_fit.x[1]
        else:
            initial_guess = np.array([np.min(y_data), np.max(y_data), 0.5], dtype=float)
            bounds = (
                np.array([1e-4, 1e-4, 0.05], dtype=float),
                np.array([np.max(y_data) + 50.0, 1e5, 1.5], dtype=float),
            )

            final_fit = fit_row_sisko_model(x_data, y_data, initial_guess, bounds)
            eta_inf_sisko = final_fit.x[0]
            k_sisko = final_fit.x[1]
            n_sisko = final_fit.x[2]

        y_pred_all = sisko_viscosity(x_data, eta_inf_sisko, k_sisko, n_sisko)
        sisko_rmse = np.sqrt(mean_squared_error(y_data, y_pred_all))
        sisko_r2 = r2_score(y_data, y_pred_all)
        sisko_mae = mean_absolute_error(y_data, y_pred_all)

        record = raw_row[input_cols].to_dict()
        record['row_index'] = row_idx
        record['Composite_Mix_ID'] = raw_row['Composite_Mix_ID']
        record['eta_inf_sisko'] = eta_inf_sisko
        record['k_sisko'] = k_sisko
        record['n_sisko'] = n_sisko
        record['SISKO_N_RMSE'] = sisko_rmse
        record['SISKO_N_R2'] = sisko_r2
        record['SISKO_N_MAE'] = sisko_mae

        sisko_dataset.append(record)
    except Exception:
        failed_fits.append(row_idx)

print(f'Successful fits : {len(sisko_dataset)}')
print(f'Failed fits     : {len(failed_fits)}')

Sisko fit mode: full 3-parameter fit
Log residuals: True
Fitting Sisko parameters for 178 rows (entire dataset)...
Successful fits : 178
Failed fits     : 0


## 6. Build Sisko Physics Dataset

In [12]:
physics_df = pd.DataFrame(sisko_dataset)
physics_df['sisko_confidence'] = 1.0 / (physics_df['SISKO_N_RMSE'] + 1e-6)

print(f'Physics dataset shape: {physics_df.shape}')
print(f"Unique formulations : {physics_df['Composite_Mix_ID'].nunique()}")
print('\nSisko parameter and fit summary:')
display(physics_df[['eta_inf_sisko', 'k_sisko', 'n_sisko', 'SISKO_N_RMSE', 'SISKO_N_R2', 'SISKO_N_MAE']].describe().round(4))

print('\nDataset preview:')
display(physics_df.head())

Physics dataset shape: (178, 19)
Unique formulations : 68

Sisko parameter and fit summary:


,eta_inf_sisko,k_sisko,n_sisko,SISKO_N_RMSE,SISKO_N_R2,SISKO_N_MAE
count,178.0000,178.0000,178.0000,178.0000,178.0000,178.0000
mean,3.4814,52.5506,0.3700,1.9762,0.9736,1.5390
std,3.1782,66.3625,0.2429,2.1004,0.0376,1.7634
min,0.0001,0.4928,0.0500,0.0080,0.6175,0.0071
25%,0.9695,5.2944,0.1576,0.2454,0.9628,0.2051
50%,2.3528,23.3103,0.3184,1.0693,0.9846,0.7209
75%,4.7748,72.4868,0.5831,2.8558,0.9945,2.0695
max,11.7295,324.6612,0.8225,7.8774,1.0000,7.0325



Dataset preview:


,Dispersent_Type,Solid_Content_pct,Solid_Additive_pct,NMC_pct,C65_pct,KS6L_pct,PVDF_pct,Source_Batch,Solid_to_Liquid_Ratio,Binder_to_Active_Ratio,row_index,Composite_Mix_ID,eta_inf_sisko,k_sisko,n_sisko,SISKO_N_RMSE,SISKO_N_R2,SISKO_N_MAE,sisko_confidence
0,Hypermer,73.0,0.50,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,0,F1_Hypermer_73.0_0.5,1.420667,9.925554,0.380070,0.450302,0.985110,0.266060,2.220728
1,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,1,F1_Hypermer_73.0_0.25,3.838806,71.603774,0.129916,2.222283,0.994349,1.565130,0.449987
2,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,2,F1_Hypermer_73.0_0.25,1.091736,9.098444,0.380028,0.313979,0.991775,0.183932,3.184913
3,Hypermer,77.0,0.25,96.0,2.0,0.0,2.0,Batch_1,3.347826,0.020408,3,F1_Hypermer_77.0_0.25,6.488646,61.004056,0.249193,3.814978,0.972666,2.912725,0.262125
4,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,4,F1_Hypermer_73.0_0.25,1.056634,7.979421,0.621578,0.467315,0.964728,0.416959,2.139878


## 7. Train-Test Split

In [13]:
unique_formulations = physics_df['Composite_Mix_ID'].dropna().unique()
form_train, form_test = train_test_split(
    unique_formulations,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

physics_train = physics_df[physics_df['Composite_Mix_ID'].isin(form_train)].copy()
physics_test = physics_df[physics_df['Composite_Mix_ID'].isin(form_test)].copy()

train_forms = set(physics_train['Composite_Mix_ID'].unique())
test_forms = set(physics_test['Composite_Mix_ID'].unique())
overlap_forms = train_forms & test_forms

print(f'Train set: {len(physics_train)} rows  |  {len(train_forms)} formulations')
print(f'Test set : {len(physics_test)} rows  |  {len(test_forms)} formulations')
print(f'Formulation overlap (must be 0): {len(overlap_forms)}')

Train set: 136 rows  |  54 formulations
Test set : 42 rows  |  14 formulations
Formulation overlap (must be 0): 0


## 8. ML Model Training

In [14]:
ml_feature_cols = input_cols
candidate_target_cols = ['eta_inf_sisko', 'k_sisko', 'n_sisko']

if USE_LOCKED_ETA_INF_ABLATION:
    locked_target_cols = ['eta_inf_sisko']
else:
    locked_target_cols = [c for c in candidate_target_cols if physics_df[c].nunique(dropna=False) <= 1]

ml_target_cols = [c for c in candidate_target_cols if c not in locked_target_cols]

X_train = physics_train[ml_feature_cols].copy()
X_test = physics_test[ml_feature_cols].copy()
y_train_raw = physics_train[ml_target_cols].copy()
y_test_raw = physics_test[ml_target_cols].copy()

TARGET_LOG_EPS = 1e-8
y_train_model = np.log(y_train_raw.clip(lower=TARGET_LOG_EPS))

sample_weight_train = physics_train['sisko_confidence'].copy()
sample_weight_train = sample_weight_train / sample_weight_train.mean()
sample_weight_train = sample_weight_train.clip(lower=0.25, upper=4.0)
sample_noise_train = 1e-4 / sample_weight_train.values

X_train_encoded = pd.get_dummies(X_train, columns=['Dispersent_Type', 'Source_Batch'], drop_first=False)
X_test_encoded = pd.get_dummies(X_test, columns=['Dispersent_Type', 'Source_Batch'], drop_first=False)

missing_cols = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0
X_test_encoded = X_test_encoded[X_train_encoded.columns]

X_train_encoded = X_train_encoded.fillna(X_train_encoded.mean())
X_test_encoded = X_test_encoded.fillna(X_train_encoded.mean())

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print(f'ML training set: {X_train.shape}')
print(f'ML test set:     {X_test.shape}')
print(f'Targets: {ml_target_cols}')
if locked_target_cols:
    print(f'Locked targets excluded: {locked_target_cols}')

ML training set: (136, 10)
ML test set:     (42, 10)
Targets: ['eta_inf_sisko', 'k_sisko', 'n_sisko']


### 8.1 Gaussian Process Regression

In [15]:
gpr_models = {}
gpr_train_r2 = {}
gpr_test_r2 = {}

for target_param in ml_target_cols:
    kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=2.5) + WhiteKernel(noise_level=0.05, noise_level_bounds=(1e-6, 1e1))
    gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, alpha=sample_noise_train, random_state=42)

    gpr.fit(X_train_scaled, y_train_model[target_param])
    gpr_models[target_param] = gpr

    y_pred_train = np.exp(gpr.predict(X_train_scaled))
    y_pred_test = np.exp(gpr.predict(X_test_scaled))

    gpr_train_r2[target_param] = r2_score(y_train_raw[target_param], y_pred_train)
    gpr_test_r2[target_param] = r2_score(y_test_raw[target_param], y_pred_test)

print('GPR Model Performance:')
print('\nTrain R2 scores:')
for param, r2 in gpr_train_r2.items():
    print(f'  {param:15s}: {r2:.4f}')

print('\nTest R2 scores:')
for param, r2 in gpr_test_r2.items():
    print(f'  {param:15s}: {r2:.4f}')

GPR Model Performance:

Train R2 scores:
  eta_inf_sisko  : 0.9365
  k_sisko        : 0.7491
  n_sisko        : 0.6486

Test R2 scores:
  eta_inf_sisko  : 0.7639
  k_sisko        : 0.6849
  n_sisko        : 0.4554


### 8.2 Random Forest and Extra Trees

In [16]:
rf_train_r2, rf_test_r2 = {}, {}
et_train_r2, et_test_r2 = {}, {}

for target_param in ml_target_cols:
    rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
    rf.fit(X_train_encoded, y_train_model[target_param], sample_weight=sample_weight_train.values)
    rf_train_r2[target_param] = r2_score(y_train_raw[target_param], np.exp(rf.predict(X_train_encoded)))
    rf_test_r2[target_param] = r2_score(y_test_raw[target_param], np.exp(rf.predict(X_test_encoded)))

    et = ExtraTreesRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
    et.fit(X_train_encoded, y_train_model[target_param], sample_weight=sample_weight_train.values)
    et_train_r2[target_param] = r2_score(y_train_raw[target_param], np.exp(et.predict(X_train_encoded)))
    et_test_r2[target_param] = r2_score(y_test_raw[target_param], np.exp(et.predict(X_test_encoded)))

comparison = pd.DataFrame({
    'Parameter': ml_target_cols,
    'GPR_Train_R2': [gpr_train_r2[p] for p in ml_target_cols],
    'GPR_Test_R2': [gpr_test_r2[p] for p in ml_target_cols],
    'RF_Train_R2': [rf_train_r2[p] for p in ml_target_cols],
    'RF_Test_R2': [rf_test_r2[p] for p in ml_target_cols],
    'ET_Train_R2': [et_train_r2[p] for p in ml_target_cols],
    'ET_Test_R2': [et_test_r2[p] for p in ml_target_cols],
})

display(comparison.round(4))

,Parameter,GPR_Train_R2,GPR_Test_R2,RF_Train_R2,RF_Test_R2,ET_Train_R2,ET_Test_R2
0,eta_inf_sisko,0.9365,0.7639,0.9703,0.8673,0.9718,0.8391
1,k_sisko,0.7491,0.6849,0.7956,0.6725,0.8095,0.6620
2,n_sisko,0.6486,0.4554,0.8172,0.3228,0.8296,0.2665
